# UNO Bayesian Curriculum Notebook

This notebook builds synthetic training curricula for Bayesian neural networks (BNNs) that model 2v2 UNO opponents across Classic and Go Wild rule sets. It covers scenario generation, persona-driven labeling, uncertainty-aware active learning, and iterative model refinement.

## Roadmap

1. Environment setup and shared utilities
2. Scenario Forge: parameterized synthetic state generation
3. Persona & Oracle bots for labeled behaviors
4. Dataset assembly and encoding
5. Bayesian neural network architectures and training
6. Active curriculum loop with uncertainty-guided self-play
7. Export trained Classic 2v2 and Go Wild 2v2 models

## Step-by-Step Training Guide

1. **Choose your runtime**
   - In Colab, open `Runtime -> Change runtime type` and select a GPU (preferred) or CPU. Confirm and restart the runtime so the environment is clean.
2. **Install dependencies**
   - Run the first code cell (`If running in Colab, install dependencies.`). Wait for `pip` to finish before moving on. Re-run this any time you restart the runtime.
3. **Initialize notebook state**
   - Execute all code cells up to `## Running the Curriculum` in order (`Runtime -> Run after` or `Runtime -> Run all`) so every helper class, encoder, and training routine is defined in memory.
4. **Review training configuration**
   - In the `## Running the Curriculum` section, inspect the hyperparameters (scenario counts, iterations, learning-rate schedules). Adjust them only if you need a longer/shorter run.
5. **Launch the curriculum loop**
   - Run the curriculum cell. Training will first build the Classic 2v2 model and then the Go Wild 2v2 model, printing progress for each phase. Expect several minutes on GPU, longer on CPU.
6. **Monitor checkpoints**
   - After each loop finishes, confirm the printed paths inside the `models/` directory. Download artifacts via the Colab file browser or use `from google.colab import files; files.download(...)` to grab local copies.
7. **Optional verification and iteration**
   - Use the `### Inference Helpers` section to reload checkpoints and sanity-check predictions. Tweak hyperparameters, restart the runtime, and repeat from step 2 if you plan additional curriculum iterations.

In [ ]:
# If running in Colab, install dependencies.
import os

IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in str(getattr(__import__("sys"), "modules", {}))
if IN_COLAB:
    try:
        import pyro  # type: ignore
    except Exception:  # pragma: no cover - installation path
        !pip install -q pyro-ppl torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu


In [ ]:
from __future__ import annotations

import abc
import dataclasses
import enum
import functools
import itertools
import json
import math
import random
import statistics
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import pyro
import pyro.distributions as dist
from pyro import poutine
from pyro.infer import SVI, Trace_ELBO
from pyro.nn import PyroModule, PyroSample, PyroParam
from pyro.optim import ClippedAdam
from pyro.distributions import constraints

from uno_engine.deck import build_deck_for_mode, shuffle_in_place
from uno_engine.engine import UnoEngine, InvalidMoveError
from uno_engine.models import (
    Card,
    CardType,
    Color,
    GameMode,
    GameState,
    PendingAction,
    PendingActionType,
    Player,
    PlayDirection,
)

RNG = random.Random(13)
torch.manual_seed(13)
np.random.seed(13)


In [ ]:
class ScenarioType(enum.Enum):
    FINISHER = "FINISHER"
    DEFENDER = "DEFENDER"
    SETUP = "SETUP"
    COLOR_TRAP = "COLOR_TRAP"
    WILD_DILEMMA = "WILD_DILEMMA"


PLAYER_SEQUENCE = ("P0", "P1", "P2", "P3")
TEAM_MAP = {"P0": 0, "P2": 0, "P1": 1, "P3": 1}
TEAM_SCORES = {0: 0, 1: 0}
STANDARD_COLORS = [Color.RED, Color.YELLOW, Color.GREEN, Color.BLUE]


@dataclass
class ScenarioParameters:
    scenario_type: ScenarioType
    mode: GameMode = GameMode.CLASSIC_2V2
    target_player: str = "P0"
    distance_to_victory: int = 1
    draw_stack_size: int = 2
    hand_diversity: int = 3
    color_bias: Optional[Color] = None
    randomize: bool = True


@dataclass
class ScenarioExample:
    state: GameState
    target_player: str
    scenario_type: ScenarioType
    parameters: ScenarioParameters
    metadata: Dict[str, Any]


class ScenarioForge:
    """Procedurally generate goal-directed UNO states for downstream training."""

    def __init__(self, *, rng: Optional[random.Random] = None) -> None:
        self.rng = rng or random.Random()

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------
    def generate(self, params: ScenarioParameters) -> ScenarioExample:
        handler = {
            ScenarioType.FINISHER: self._finisher,
            ScenarioType.DEFENDER: self._defender,
            ScenarioType.SETUP: self._setup,
            ScenarioType.COLOR_TRAP: self._color_trap,
            ScenarioType.WILD_DILEMMA: self._wild_dilemma,
        }[params.scenario_type]
        return handler(params)

    def generate_batch(
        self,
        scenario_mix: Dict[ScenarioType, float],
        *,
        mode: GameMode,
        batch_size: int,
        base_params: Optional[ScenarioParameters] = None,
    ) -> List[ScenarioExample]:
        weights = np.array(list(scenario_mix.values()), dtype=np.float64)
        if not math.isclose(weights.sum(), 1.0):
            weights = weights / weights.sum()
        scenarios = list(scenario_mix.keys())

        examples: List[ScenarioExample] = []
        for _ in range(batch_size):
            scenario_type = self.rng.choices(scenarios, weights=weights)[0]
            params = dataclasses.replace(
                base_params or ScenarioParameters(scenario_type=scenario_type, mode=mode),
                scenario_type=scenario_type,
                mode=mode,
            )
            examples.append(self.generate(params))
        return examples

    # ------------------------------------------------------------------
    # Scenario builders
    # ------------------------------------------------------------------
    def _finisher(self, params: ScenarioParameters) -> ScenarioExample:
        distance = max(1, params.distance_to_victory)
        top_color = params.color_bias or self.rng.choice(STANDARD_COLORS)
        top_number = self.rng.randint(0, 9)
        top_card = self._make_number_card(top_color, top_number)

        target_hand = self._make_random_hand(params.mode, hand_size=distance)
        playable_card = self._ensure_playable(target_hand, top_card, top_color)

        other_hands = {
            pid: self._make_random_hand(params.mode, hand_size=self.rng.randint(4, 7))
            for pid in PLAYER_SEQUENCE
            if pid != params.target_player
        }

        current_player_idx = PLAYER_SEQUENCE.index(params.target_player)
        state = self._assemble_state(
            mode=params.mode,
            hands={params.target_player: target_hand, **other_hands},
            discard=[top_card],
            current_player_index=current_player_idx,
            current_color=top_color,
        )

        metadata = {
            "playable_card": playable_card,
            "distance_to_victory": distance,
        }
        return ScenarioExample(state, params.target_player, ScenarioType.FINISHER, params, metadata)

    def _defender(self, params: ScenarioParameters) -> ScenarioExample:
        stack_size = max(2, params.draw_stack_size)
        stack_card_type = CardType.DRAW_TWO if stack_size % 4 != 0 else CardType.WILD_DRAW_FOUR
        top_card = (
            self._make_action_card(self.rng.choice(STANDARD_COLORS), CardType.DRAW_TWO)
            if stack_card_type == CardType.DRAW_TWO
            else self._make_wild_draw_four()
        )

        target_hand = self._make_random_hand(params.mode, hand_size=self.rng.randint(3, 6))
        response_card = self._ensure_stack_response(target_hand, stack_card_type)

        other_hands = {
            pid: self._make_random_hand(params.mode, hand_size=self.rng.randint(4, 7))
            for pid in PLAYER_SEQUENCE
            if pid != params.target_player
        }

        pending = PendingAction(
            player_id=params.target_player,
            type=PendingActionType.DRAW_STACK,
            allowed_cards=tuple(
                card
                for card in target_hand
                if card.type in {CardType.DRAW_TWO, CardType.WILD_DRAW_FOUR}
            ),
            draw_penalty=stack_size,
        )

        state = self._assemble_state(
            mode=params.mode,
            hands={params.target_player: target_hand, **other_hands},
            discard=[top_card],
            current_player_index=PLAYER_SEQUENCE.index(params.target_player),
            current_color=top_card.color if top_card.color != Color.WILD else None,
            pending_action=pending,
            draw_stack_total=stack_size,
        )

        metadata = {
            "draw_stack_penalty": stack_size,
            "response_card": response_card,
        }
        return ScenarioExample(state, params.target_player, ScenarioType.DEFENDER, params, metadata)

    def _setup(self, params: ScenarioParameters) -> ScenarioExample:
        diversity = max(2, min(4, params.hand_diversity))
        preferred_colors = self.rng.sample(STANDARD_COLORS, diversity)
        teammate = self._teammate_of(params.target_player)
        teammate_color = preferred_colors[0]

        top_color = params.color_bias or preferred_colors[-1]
        top_card = self._make_number_card(top_color, self.rng.randint(0, 9))

        target_hand = []
        for color in preferred_colors:
            target_hand.append(self._make_number_card(color, self.rng.randint(0, 9)))
        target_hand.append(self._make_action_card(preferred_colors[0], CardType.REVERSE))
        target_hand.append(self._make_wild())

        teammate_hand = [self._make_number_card(teammate_color, self.rng.randint(1, 9)) for _ in range(5)]
        opponent_hands = {
            pid: self._make_random_hand(params.mode, hand_size=self.rng.randint(5, 8))
            for pid in PLAYER_SEQUENCE
            if pid not in {params.target_player, teammate}
        }

        state = self._assemble_state(
            mode=params.mode,
            hands={
                params.target_player: target_hand,
                teammate: teammate_hand,
                **opponent_hands,
            },
            discard=[top_card],
            current_player_index=PLAYER_SEQUENCE.index(params.target_player),
            current_color=top_color,
        )

        metadata = {
            "teammate_preferred_color": teammate_color,
            "available_wild": True,
        }
        return ScenarioExample(state, params.target_player, ScenarioType.SETUP, params, metadata)

    def _color_trap(self, params: ScenarioParameters) -> ScenarioExample:
        weak_color = params.color_bias or self.rng.choice(STANDARD_COLORS)
        strong_color = self.rng.choice([c for c in STANDARD_COLORS if c != weak_color])
        top_card = self._make_number_card(strong_color, self.rng.randint(0, 9))

        target_hand = self._make_random_hand(params.mode, hand_size=self.rng.randint(5, 7))
        target_hand.append(self._make_action_card(strong_color, CardType.SKIP))

        teammate = self._teammate_of(params.target_player)
        teammate_hand = self._make_random_hand(params.mode, hand_size=self.rng.randint(4, 6))

        opponents = [pid for pid in PLAYER_SEQUENCE if pid not in {params.target_player, teammate}]
        opponent_hands = {
            opponents[0]: self._make_hand_without_color(params.mode, color=weak_color, hand_size=self.rng.randint(5, 7)),
            opponents[1]: self._make_random_hand(params.mode, hand_size=self.rng.randint(5, 7)),
        }

        state = self._assemble_state(
            mode=params.mode,
            hands={
                params.target_player: target_hand,
                teammate: teammate_hand,
                **opponent_hands,
            },
            discard=[top_card],
            current_player_index=PLAYER_SEQUENCE.index(params.target_player),
            current_color=top_card.color,
        )

        metadata = {
            "weak_opponent_color": weak_color.value,
            "target_color": strong_color.value,
        }
        return ScenarioExample(state, params.target_player, ScenarioType.COLOR_TRAP, params, metadata)

    def _wild_dilemma(self, params: ScenarioParameters) -> ScenarioExample:
        base_color = params.color_bias or self.rng.choice(STANDARD_COLORS)
        alt_color = self.rng.choice([c for c in STANDARD_COLORS if c != base_color])
        top_card = self._make_number_card(base_color, self.rng.randint(1, 9))

        target_hand = [
            self._make_number_card(base_color, top_card.number),
            self._make_number_card(alt_color, self.rng.randint(0, 9)),
            self._make_action_card(base_color, CardType.SKIP),
            self._make_wild(),
            self._make_wild_draw_four() if params.mode == GameMode.GO_WILD_2V2 else self._make_number_card(base_color, self.rng.randint(0, 9)),
        ]
        other_hands = {
            pid: self._make_random_hand(params.mode, hand_size=self.rng.randint(4, 7))
            for pid in PLAYER_SEQUENCE
            if pid != params.target_player
        }

        state = self._assemble_state(
            mode=params.mode,
            hands={params.target_player: target_hand, **other_hands},
            discard=[top_card],
            current_player_index=PLAYER_SEQUENCE.index(params.target_player),
            current_color=top_card.color,
        )

        metadata = {
            "normal_play_option": target_hand[0],
            "wild_play_option": target_hand[-2],
            "high_stakes": params.mode == GameMode.GO_WILD_2V2,
        }
        return ScenarioExample(state, params.target_player, ScenarioType.WILD_DILEMMA, params, metadata)

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------
    def _make_random_hand(self, mode: GameMode, hand_size: int) -> List[Card]:
        hand: List[Card] = []
        for _ in range(hand_size):
            card_type = self.rng.choices(
                [CardType.NUMBER, CardType.SKIP, CardType.REVERSE, CardType.DRAW_TWO, CardType.WILD, CardType.WILD_DRAW_FOUR, CardType.DISCARD_ALL],
                weights=[0.55, 0.08, 0.08, 0.08, 0.12, 0.05, 0.04 if mode == GameMode.GO_WILD_2V2 else 0.0],
            )[0]
            if card_type == CardType.NUMBER:
                color = self.rng.choice(STANDARD_COLORS)
                hand.append(self._make_number_card(color, self.rng.randint(0, 9)))
            elif card_type in {CardType.SKIP, CardType.REVERSE, CardType.DRAW_TWO}:
                color = self.rng.choice(STANDARD_COLORS)
                hand.append(self._make_action_card(color, card_type))
            elif card_type == CardType.WILD:
                hand.append(self._make_wild())
            elif card_type == CardType.WILD_DRAW_FOUR:
                hand.append(self._make_wild_draw_four())
            else:  # DISCARD_ALL
                color = self.rng.choice(STANDARD_COLORS)
                hand.append(self._make_action_card(color, CardType.DISCARD_ALL))
        return hand

    def _make_hand_without_color(self, mode: GameMode, color: Color, hand_size: int) -> List[Card]:
        hand = []
        while len(hand) < hand_size:
            card = self._make_random_hand(mode, 1)[0]
            if card.color != color:
                hand.append(card)
        return hand

    def _assemble_state(
        self,
        *,
        mode: GameMode,
        hands: Dict[str, Sequence[Card]],
        discard: List[Card],
        current_player_index: int,
        current_color: Optional[Color],
        pending_action: Optional[PendingAction] = None,
        draw_stack_total: int = 0,
    ) -> GameState:
        players = [
            Player(player_id=pid, hand=list(hands[pid]), score=0)
            for pid in PLAYER_SEQUENCE
        ]

        deck = build_deck_for_mode(mode)
        for card in itertools.chain.from_iterable(hands.values()):
            self._try_remove_card(deck, card)
        for card in discard:
            self._try_remove_card(deck, card)
        shuffle_in_place(deck, self.rng)

        state = GameState(
            players=players,
            draw_pile=deck,
            discard_pile=list(discard),
            current_player_index=current_player_index,
            play_direction=PlayDirection.CLOCKWISE,
            current_color=current_color,
            pending_action=pending_action,
            mode=mode,
            team_map=dict(TEAM_MAP),
            team_scores=dict(TEAM_SCORES),
            draw_stack_total=draw_stack_total,
        )
        return state

    def _try_remove_card(self, deck: List[Card], card: Card) -> None:
        for idx, candidate in enumerate(deck):
            if candidate == card:
                deck.pop(idx)
                return

    def _ensure_playable(self, hand: List[Card], top_card: Card, current_color: Color) -> Card:
        for card in hand:
            if self._is_playable(card, top_card, current_color):
                return card
        replacement = self._make_number_card(current_color, top_card.number)
        hand[0] = replacement
        return replacement

    def _ensure_stack_response(self, hand: List[Card], stack_type: CardType) -> Card:
        for card in hand:
            if card.type == stack_type or (card.type == CardType.WILD_DRAW_FOUR and stack_type == CardType.DRAW_TWO):
                return card
        if stack_type == CardType.DRAW_TWO:
            response = self._make_action_card(self.rng.choice(STANDARD_COLORS), CardType.DRAW_TWO)
        else:
            response = self._make_wild_draw_four()
        hand[0] = response
        return response

    def _is_playable(self, card: Card, top_card: Card, current_color: Color) -> bool:
        if card.type == CardType.WILD:
            return True
        if card.type == CardType.WILD_DRAW_FOUR:
            return True
        if card.color == current_color:
            return True
        if card.type == top_card.type:
            return True
        if card.type == CardType.NUMBER and top_card.type == CardType.NUMBER:
            return card.number == top_card.number
        return False

    def _make_number_card(self, color: Color, number: int) -> Card:
        return Card(color=color, type=CardType.NUMBER, number=number, value=number)

    def _make_action_card(self, color: Color, card_type: CardType) -> Card:
        value_map = {
            CardType.SKIP: 20,
            CardType.REVERSE: 20,
            CardType.DRAW_TWO: 20,
            CardType.DISCARD_ALL: 40,
        }
        return Card(color=color, type=card_type, value=value_map[card_type])

    def _make_wild(self) -> Card:
        return Card(color=Color.WILD, type=CardType.WILD, value=50)

    def _make_wild_draw_four(self) -> Card:
        return Card(color=Color.WILD, type=CardType.WILD_DRAW_FOUR, value=50)

    def _teammate_of(self, player_id: str) -> str:
        team_index = TEAM_MAP[player_id]
        for pid, t_index in TEAM_MAP.items():
            if pid != player_id and t_index == team_index:
                return pid
        raise ValueError(f"No teammate registered for player '{player_id}'")


In [ ]:
class ActionType(enum.Enum):
    PLAY = "PLAY"
    DRAW = "DRAW"
    PASS = "PASS"


@dataclass
class BotAction:
    action_type: ActionType
    card: Optional[Card] = None
    chosen_color: Optional[Color] = None
    info: Dict[str, Any] = field(default_factory=dict)


class BotPolicy(abc.ABC):
    def __init__(self, name: str, rng: Optional[random.Random] = None) -> None:
        self.name = name
        self.rng = rng or random.Random()

    def enumerate_actions(self, engine: UnoEngine, state: GameState, player_id: str) -> List[BotAction]:
        valid_cards = engine.get_valid_moves(state, player_id)
        actions: List[BotAction] = []
        for card in valid_cards:
            if card.type in {CardType.WILD, CardType.WILD_DRAW_FOUR}:
                for color in STANDARD_COLORS:
                    actions.append(BotAction(ActionType.PLAY, card=card, chosen_color=color))
            else:
                actions.append(BotAction(ActionType.PLAY, card=card))
        if not actions:
            pending = state.pending_action
            if pending and pending.type == PendingActionType.DRAW_STACK:
                actions.append(BotAction(ActionType.DRAW))
            else:
                actions.append(BotAction(ActionType.DRAW))
        return actions

    @abc.abstractmethod
    def decide(self, engine: UnoEngine, state: GameState, player_id: str) -> BotAction:
        ...

    # Utility scoring helpers -------------------------------------------------
    def teammate_id(self, state: GameState, player_id: str) -> str:
        return state.teammate_id(player_id)

    def team_indices(self, state: GameState, player_id: str) -> Tuple[int, int]:
        my_team = state.team_index_for(player_id)
        return my_team, 1 - my_team

    def color_histogram(self, cards: Sequence[Card]) -> Dict[Color, int]:
        counts: Dict[Color, int] = {color: 0 for color in STANDARD_COLORS}
        for card in cards:
            if card.color in counts:
                counts[card.color] += 1
        return counts

    def choose_color_for_teammate(self, teammate_hand: Sequence[Card]) -> Color:
        counts = self.color_histogram(teammate_hand)
        return max(counts.items(), key=lambda kv: kv[1])[0]


class OracleBot(BotPolicy):
    def __init__(self, *, rollout_count: int = 16, rollout_depth: int = 6, rng: Optional[random.Random] = None) -> None:
        super().__init__("OracleBot", rng=rng)
        self.rollout_count = rollout_count
        self.rollout_depth = rollout_depth
        self.engine = UnoEngine()

    def decide(self, engine: UnoEngine, state: GameState, player_id: str) -> BotAction:
        actions = self.enumerate_actions(engine, state, player_id)
        if len(actions) == 1:
            return actions[0]

        scores: List[float] = []
        for action in actions:
            estimate = self._estimate_value(state, player_id, action)
            scores.append(estimate)
        best_idx = int(np.argmax(scores))
        best_action = actions[best_idx]
        best_action.info["value_estimate"] = scores[best_idx]
        return best_action

    def _estimate_value(self, state: GameState, player_id: str, action: BotAction) -> float:
        total = 0.0
        for _ in range(self.rollout_count):
            simulated = self._apply_action(state, player_id, action)
            total += self._rollout_value(simulated, player_id)
        return total / self.rollout_count

    def _apply_action(self, state: GameState, player_id: str, action: BotAction) -> GameState:
        if action.action_type == ActionType.PLAY and action.card is not None:
            return self.engine.play_card(state, player_id, action.card, action.chosen_color)
        if action.action_type == ActionType.DRAW:
            try:
                return self.engine.draw_card(state, player_id)
            except InvalidMoveError:
                return state
        if action.action_type == ActionType.PASS:
            try:
                return self.engine.pass_turn(state, player_id)
            except InvalidMoveError:
                return state
        return state

    def _rollout_value(self, state: GameState, origin_player: str) -> float:
        working_state = state
        depth = self.rollout_depth
        current_engine = self.engine
        while depth > 0 and not working_state.round_over:
            current_player = working_state.current_player().player_id
            actions = self.enumerate_actions(current_engine, working_state, current_player)
            choice = self.rng.choice(actions)
            working_state = self._apply_action(working_state, current_player, choice)
            depth -= 1
        return self._heuristic_value(working_state, origin_player)

    def _heuristic_value(self, state: GameState, player_id: str) -> float:
        my_team, opp_team = self.team_indices(state, player_id)
        my_cards = sum(len(p.hand) for p in state.players if state.team_map[p.player_id] == my_team)
        opp_cards = sum(len(p.hand) for p in state.players if state.team_map[p.player_id] == opp_team)
        score = opp_cards - my_cards
        if state.round_over:
            if state.team_index_for(state.round_winner_id or player_id) == my_team:
                score += 50
            else:
                score -= 50
        return score


class AggressorBot(BotPolicy):
    def __init__(self, rng: Optional[random.Random] = None) -> None:
        super().__init__("AggressorBot", rng=rng)

    def decide(self, engine: UnoEngine, state: GameState, player_id: str) -> BotAction:
        actions = self.enumerate_actions(engine, state, player_id)
        priority = {
            CardType.WILD_DRAW_FOUR: 5,
            CardType.DRAW_TWO: 4,
            CardType.SKIP: 3,
            CardType.REVERSE: 2,
        }
        best = actions[0]
        best_score = -float("inf")
        for action in actions:
            score = 0.0
            if action.action_type != ActionType.PLAY or action.card is None:
                continue
            score += priority.get(action.card.type, 0)
            if action.card.type in {CardType.WILD, CardType.WILD_DRAW_FOUR} and action.chosen_color is not None:
                score += self._penalty_color_bonus(state, player_id, action.chosen_color)
            if score > best_score:
                best_score = score
                best = action
        return best

    def _penalty_color_bonus(self, state: GameState, player_id: str, color: Color) -> float:
        my_team, opp_team = self.team_indices(state, player_id)
        opp_color_counts = 0
        for player in state.players:
            if state.team_map[player.player_id] == opp_team:
                opp_color_counts += sum(1 for c in player.hand if c.color == color)
        return -0.1 * opp_color_counts


class SupporterBot(BotPolicy):
    def __init__(self, rng: Optional[random.Random] = None) -> None:
        super().__init__("SupporterBot", rng=rng)

    def decide(self, engine: UnoEngine, state: GameState, player_id: str) -> BotAction:
        actions = self.enumerate_actions(engine, state, player_id)
        teammate = self.teammate_id(state, player_id)
        teammate_hand = next(p.hand for p in state.players if p.player_id == teammate)
        preferred_color = self.choose_color_for_teammate(teammate_hand)

        scored: List[Tuple[float, BotAction]] = []
        for action in actions:
            score = 0.0
            if action.action_type == ActionType.PLAY and action.card is not None:
                if action.card.color == preferred_color:
                    score += 2.0
                if action.card.type in {CardType.SKIP, CardType.REVERSE}:
                    score += 1.0
                if action.card.type == CardType.WILD and action.chosen_color == preferred_color:
                    score += 3.0
                if action.card.type == CardType.WILD_DRAW_FOUR and action.chosen_color == preferred_color:
                    score += 2.5
                if action.card.type == CardType.DISCARD_ALL and action.card.color == preferred_color:
                    score += 2.0
            scored.append((score, action))
        scored.sort(key=lambda pair: pair[0], reverse=True)
        return scored[0][1]


class ConservativeBot(BotPolicy):
    def __init__(self, rng: Optional[random.Random] = None) -> None:
        super().__init__("ConservativeBot", rng=rng)

    def decide(self, engine: UnoEngine, state: GameState, player_id: str) -> BotAction:
        actions = self.enumerate_actions(engine, state, player_id)
        ranked: List[Tuple[float, BotAction]] = []
        for action in actions:
            if action.action_type != ActionType.PLAY or action.card is None:
                ranked.append((float("inf"), action))
                continue
            card_value = action.card.value
            penalty = 0.0
            if action.card.type in {CardType.WILD_DRAW_FOUR, CardType.DRAW_TWO}:
                penalty += 3.0
            if action.card.type == CardType.WILD:
                penalty += 1.5
            ranked.append((card_value + penalty, action))
        ranked.sort(key=lambda pair: pair[0])
        return ranked[0][1]


class RandomBot(BotPolicy):
    def __init__(self, rng: Optional[random.Random] = None) -> None:
        super().__init__("RandomBot", rng=rng)

    def decide(self, engine: UnoEngine, state: GameState, player_id: str) -> BotAction:
        actions = self.enumerate_actions(engine, state, player_id)
        return self.rng.choice(actions)


In [ ]:
@dataclass
class LabeledScenario:
    scenario: ScenarioExample
    bot_name: str
    action: BotAction
    resulting_state: GameState
    action_success: bool
    info: Dict[str, Any] = field(default_factory=dict)


class ScenarioLabeler:
    def __init__(self, *, rng: Optional[random.Random] = None) -> None:
        self.engine = UnoEngine()
        self.rng = rng or random.Random()

    def label(
        self,
        scenarios: Sequence[ScenarioExample],
        bots: Sequence[BotPolicy],
    ) -> List[LabeledScenario]:
        labeled: List[LabeledScenario] = []
        for scenario in scenarios:
            for bot in bots:
                action = bot.decide(self.engine, scenario.state, scenario.target_player)
                next_state, success = self._apply_action(scenario.state, scenario.target_player, action)
                info = {"bot_decision_metadata": action.info}
                labeled.append(
                    LabeledScenario(
                        scenario=scenario,
                        bot_name=bot.name,
                        action=action,
                        resulting_state=next_state,
                        action_success=success,
                        info=info,
                    )
                )
        return labeled

    def logs_to_labeled(self, logs: Sequence[ActiveLog]) -> List[LabeledScenario]:
        converted: List[LabeledScenario] = []
        for log in logs:
            parameters = ScenarioParameters(
                scenario_type=log.scenario_type,
                mode=log.state.mode,
                target_player=log.target_player,
            )
            scenario = ScenarioExample(
                state=log.state,
                target_player=log.target_player,
                scenario_type=log.scenario_type,
                parameters=parameters,
                metadata=log.metadata,
            )
            converted.append(
                LabeledScenario(
                    scenario=scenario,
                    bot_name=log.persona_name,
                    action=log.action,
                    resulting_state=log.resulting_state,
                    action_success=True,
                    info={
                        "bnn_entropy": log.bnn_entropy,
                        "bnn_mutual_information": log.bnn_mutual_information,
                    },
                )
            )
        return converted

    def _apply_action(
        self,
        state: GameState,
        player_id: str,
        action: BotAction,
    ) -> Tuple[GameState, bool]:
        try:
            if action.action_type == ActionType.PLAY and action.card is not None:
                next_state = self.engine.play_card(state, player_id, action.card, action.chosen_color)
            elif action.action_type == ActionType.DRAW:
                next_state = self.engine.draw_card(state, player_id)
            elif action.action_type == ActionType.PASS:
                next_state = self.engine.pass_turn(state, player_id)
            else:
                next_state = state
            return next_state, True
        except InvalidMoveError:
            return state, False


In [ ]:
# Quick sanity check: generate one sample per archetype and preview bot actions.
forge = ScenarioForge(rng=random.Random(21))
labeler = ScenarioLabeler(rng=random.Random(22))
bots = [OracleBot(rng=random.Random(23)), AggressorBot(rng=random.Random(24))]

samples: Dict[str, Dict[str, Any]] = {}
for scenario_type in ScenarioType:
    scenario = forge.generate(ScenarioParameters(scenario_type=scenario_type))
    labeled = labeler.label([scenario], bots)
    samples[scenario_type.value] = {
        "target_player": scenario.target_player,
        "top_discard": scenario.state.discard_pile[-1].type.name,
        "bot_actions": {entry.bot_name: entry.action.info for entry in labeled},
    }
samples

In [ ]:
CARD_TYPES_FOR_ENCODING = [
    CardType.NUMBER,
    CardType.SKIP,
    CardType.REVERSE,
    CardType.DRAW_TWO,
    CardType.WILD,
    CardType.WILD_DRAW_FOUR,
    CardType.DISCARD_ALL,
]
SPECIAL_TYPES = [
    CardType.SKIP,
    CardType.REVERSE,
    CardType.DRAW_TWO,
    CardType.WILD,
    CardType.WILD_DRAW_FOUR,
    CardType.DISCARD_ALL,
]
PENDING_TYPES = [PendingActionType.DRAWN_CARD_PLAY_WINDOW, PendingActionType.COLOR_CHOICE, PendingActionType.DRAW_STACK]
BOT_ORDER = ["OracleBot", "AggressorBot", "SupporterBot", "ConservativeBot", "RandomBot"]
SCENARIO_ORDER = list(ScenarioType)
MODE_ORDER = [GameMode.CLASSIC_2V2, GameMode.GO_WILD_2V2]


class StateEncoder:
    def __init__(self) -> None:
        self.feature_size = self._compute_feature_size()

    def encode(self, labeled: LabeledScenario) -> np.ndarray:
        return self.encode_components(
            state=labeled.scenario.state,
            target_player=labeled.scenario.target_player,
            bot_name=labeled.bot_name,
            scenario_type=labeled.scenario.scenario_type,
            metadata=labeled.scenario.metadata,
        )

    def encode_components(
        self,
        *,
        state: GameState,
        target_player: str,
        bot_name: str,
        scenario_type: ScenarioType,
        metadata: Optional[Dict[str, Any]] = None,
    ) -> np.ndarray:
        metadata = metadata or {}
        features: List[float] = []

        # Player-centric features (ordered P0..P3)
        for pid in PLAYER_SEQUENCE:
            hand = next(player.hand for player in state.players if player.player_id == pid)
            features.append(len(hand) / 20.0)
            color_counts = {color: 0 for color in STANDARD_COLORS}
            special_counts = {stype: 0 for stype in SPECIAL_TYPES}
            numeric_sum = 0
            for card in hand:
                if card.color in color_counts:
                    color_counts[card.color] += 1
                if card.type in special_counts:
                    special_counts[card.type] += 1
                if card.type == CardType.NUMBER and card.number is not None:
                    numeric_sum += card.number
            features.extend(color_counts[color] / 10.0 for color in STANDARD_COLORS)
            features.extend(special_counts[stype] / 5.0 for stype in SPECIAL_TYPES)
            features.append(numeric_sum / 40.0)

        # Discard / top card
        top_card = state.discard_pile[-1]
        features.extend(self._encode_color(top_card.color))
        features.extend(self._encode_card_type(top_card.type))
        features.append((top_card.number or 0) / 9.0)
        features.append(1.0 if top_card.type == CardType.NUMBER else 0.0)

        # Current color context
        features.extend(self._encode_color(state.current_color))

        # Pending action signals
        pending = state.pending_action
        features.extend(self._encode_pending_type(pending.type if pending else None))
        draw_penalty = pending.draw_penalty if pending else 0
        features.append(draw_penalty / 12.0)
        features.append((len(pending.allowed_cards) if pending else 0) / 5.0)
        features.append(state.draw_stack_total / 12.0)

        # Scenario info
        features.extend(self._encode_scenario_type(scenario_type))
        features.append(metadata.get("distance_to_victory", 0) / 10.0)
        features.append(metadata.get("draw_stack_penalty", 0) / 12.0)
        features.append(1.0 if metadata.get("available_wild") else 0.0)
        features.append(1.0 if metadata.get("high_stakes") else 0.0)
        features.extend(self._encode_color(metadata.get("teammate_preferred_color")))
        features.extend(self._encode_color(metadata.get("weak_opponent_color")))

        # Target player & persona identity
        target_one_hot = [1.0 if pid == target_player else 0.0 for pid in PLAYER_SEQUENCE]
        features.extend(target_one_hot)
        bot_one_hot = [1.0 if bot_name == name else 0.0 for name in BOT_ORDER]
        features.extend(bot_one_hot)

        # Mode
        features.extend(self._encode_mode(state.mode))

        return np.array(features, dtype=np.float32)

    def _compute_feature_size(self) -> int:
        dummy_state = ScenarioForge(rng=random.Random(0)).generate(
            ScenarioParameters(scenario_type=ScenarioType.FINISHER)
        )
        dummy_labeled = LabeledScenario(
            scenario=dummy_state,
            bot_name="OracleBot",
            action=BotAction(ActionType.DRAW),
            resulting_state=dummy_state.state,
            action_success=True,
        )
        return len(self.encode(dummy_labeled))

    def _encode_color(self, color: Optional[Color]) -> List[float]:
        if isinstance(color, str):
            try:
                color = Color[color]
            except KeyError:
                color = None
        bucket = [0.0] * (len(STANDARD_COLORS) + 1)
        if color in STANDARD_COLORS:
            bucket[STANDARD_COLORS.index(color)] = 1.0
        elif color == Color.WILD:
            bucket[-1] = 1.0
        return bucket

    def _encode_card_type(self, card_type: CardType) -> List[float]:
        bucket = [0.0] * len(CARD_TYPES_FOR_ENCODING)
        if card_type in CARD_TYPES_FOR_ENCODING:
            bucket[CARD_TYPES_FOR_ENCODING.index(card_type)] = 1.0
        return bucket

    def _encode_pending_type(self, pending_type: Optional[PendingActionType]) -> List[float]:
        bucket = [0.0] * len(PENDING_TYPES)
        if pending_type in PENDING_TYPES:
            bucket[PENDING_TYPES.index(pending_type)] = 1.0
        return bucket

    def _encode_scenario_type(self, scenario_type: ScenarioType) -> List[float]:
        bucket = [0.0] * len(SCENARIO_ORDER)
        bucket[SCENARIO_ORDER.index(scenario_type)] = 1.0
        return bucket

    def _encode_mode(self, mode: GameMode) -> List[float]:
        bucket = [0.0] * len(MODE_ORDER)
        if mode in MODE_ORDER:
            bucket[MODE_ORDER.index(mode)] = 1.0
        return bucket


class ActionEncoder:
    def __init__(self) -> None:
        self.lookup: Dict[str, int] = {}
        self.reverse: Dict[int, str] = {}
        for token in self._enumerate_tokens():
            self.lookup[token] = len(self.lookup)
            self.reverse[self.lookup[token]] = token

    def encode(self, labeled: LabeledScenario) -> int:
        token = self._to_token(labeled.action)
        return self.lookup.get(token, self.lookup["DRAW"])

    def decode(self, index: int) -> str:
        return self.reverse[index]

    def token_to_action(self, token: str, state: GameState, player_id: str) -> BotAction:
        token = token.upper()
        if token == "DRAW":
            return BotAction(ActionType.DRAW)
        if token == "PASS":
            return BotAction(ActionType.PASS)
        hand = next(player.hand for player in state.players if player.player_id == player_id)
        parts = token.split("_")
        if len(parts) < 3:
            return BotAction(ActionType.DRAW)
        if parts[1] == "NUMBER" and len(parts) == 4:
            color = Color[parts[2]]
            number = int(parts[3])
            card = self._find_card(hand, lambda c: c.type == CardType.NUMBER and c.color == color and c.number == number)
            if card:
                return BotAction(ActionType.PLAY, card=card)
        elif parts[1] in {"SKIP", "REVERSE", "DRAW", "DISCARD"}:
            card_type = CardType["_".join(parts[1:len(parts)-1]) if parts[1] == "DRAW" and parts[2] == "TWO" else parts[1]]
            if card_type == CardType.DRAW_TWO:
                color = Color[parts[3]]
            elif card_type == CardType.DISCARD_ALL:
                color = Color[parts[3]]
            else:
                color = Color[parts[2]]
            card = self._find_card(hand, lambda c: c.type == card_type and c.color == color)
            if card:
                return BotAction(ActionType.PLAY, card=card)
        elif parts[1] == "WILD" and len(parts) == 3:
            chosen_color = Color[parts[2]]
            card = self._find_card(hand, lambda c: c.type == CardType.WILD)
            if card:
                return BotAction(ActionType.PLAY, card=card, chosen_color=chosen_color)
        elif parts[1] == "WILD" and parts[2] == "DRAW" and len(parts) == 5:
            chosen_color = Color[parts[4]]
            card = self._find_card(hand, lambda c: c.type == CardType.WILD_DRAW_FOUR)
            if card:
                return BotAction(ActionType.PLAY, card=card, chosen_color=chosen_color)
        return BotAction(ActionType.DRAW)

    def _find_card(self, hand: Sequence[Card], predicate: Callable[[Card], bool]) -> Optional[Card]:
        for card in hand:
            if predicate(card):
                return card
        return None

    def _to_token(self, action: BotAction) -> str:
        if action.action_type == ActionType.DRAW:
            return "DRAW"
        if action.action_type == ActionType.PASS:
            return "PASS"
        card = action.card
        if card is None:
            return "UNKNOWN"
        if card.type == CardType.NUMBER:
            return f"PLAY_NUMBER_{card.color.value}_{card.number}"
        if card.type in {CardType.SKIP, CardType.REVERSE, CardType.DRAW_TWO, CardType.DISCARD_ALL}:
            return f"PLAY_{card.type.name}_{card.color.value}"
        if card.type == CardType.WILD:
            chosen = action.chosen_color.value if action.chosen_color else "NONE"
            return f"PLAY_WILD_{chosen}"
        if card.type == CardType.WILD_DRAW_FOUR:
            chosen = action.chosen_color.value if action.chosen_color else "NONE"
            return f"PLAY_WILD_DRAW_FOUR_{chosen}"
        return "UNKNOWN"

    def _enumerate_tokens(self) -> Iterable[str]:
        yield "DRAW"
        yield "PASS"
        for color in STANDARD_COLORS:
            for number in range(0, 10):
                yield f"PLAY_NUMBER_{color.value}_{number}"
            for card_type in [CardType.SKIP, CardType.REVERSE, CardType.DRAW_TWO, CardType.DISCARD_ALL]:
                yield f"PLAY_{card_type.name}_{color.value}"
        for color in STANDARD_COLORS:
            yield f"PLAY_WILD_{color.value}"
            yield f"PLAY_WILD_DRAW_FOUR_{color.value}"


In [ ]:
class ScenarioDataset(Dataset):
    def __init__(self, features: np.ndarray, labels: np.ndarray, metadata: List[Dict[str, Any]]) -> None:
        self.features = torch.from_numpy(features).float()
        self.labels = torch.from_numpy(labels).long()
        self.metadata = metadata

    def __len__(self) -> int:  # pragma: no cover - trivial
        return len(self.features)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:  # pragma: no cover - trivial
        return self.features[index], self.labels[index]


In [ ]:
class StateBayesianNN(PyroModule):
    def __init__(self, input_dim: int, num_classes: int, hidden_dim: int = 256, dropout: float = 0.1) -> None:
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.classifier = PyroModule[nn.Linear](hidden_dim // 2, num_classes)
        self.classifier.weight = PyroSample(
            dist.Normal(torch.zeros(num_classes, hidden_dim // 2), torch.ones(num_classes, hidden_dim // 2)).to_event(2)
        )
        self.classifier.bias = PyroSample(
            dist.Normal(torch.zeros(num_classes), torch.ones(num_classes)).to_event(1)
        )

    def forward(self, x: torch.Tensor, y: Optional[torch.Tensor] = None) -> torch.Tensor:
        pyro.module("backbone", self.backbone)
        features = self.backbone(x)
        logits = self.classifier(features)
        log_probs = F.log_softmax(logits, dim=-1)
        pyro.deterministic("logits", logits)
        with pyro.plate("data", x.size(0)):
            pyro.sample("obs", dist.Categorical(logits=log_probs), obs=y)
        return log_probs


class StateBNNGuide(PyroModule):
    def __init__(self, model: StateBayesianNN) -> None:
        super().__init__()
        hidden_dim = model.classifier.in_features
        num_classes = model.classifier.out_features
        self.backbone = model.backbone
        self.weight_loc = PyroParam(torch.zeros(num_classes, hidden_dim))
        self.weight_scale = PyroParam(torch.ones(num_classes, hidden_dim), constraint=constraints.positive)
        self.bias_loc = PyroParam(torch.zeros(num_classes))
        self.bias_scale = PyroParam(torch.ones(num_classes), constraint=constraints.positive)

    def forward(self, x: torch.Tensor, y: Optional[torch.Tensor] = None) -> None:
        pyro.module("backbone", self.backbone)
        pyro.sample(
            "classifier.weight",
            dist.Normal(self.weight_loc, self.weight_scale).to_event(2),
        )
        pyro.sample(
            "classifier.bias",
            dist.Normal(self.bias_loc, self.bias_scale).to_event(1),
        )


In [ ]:
@dataclass
class TrainingConfig:
    num_epochs: int = 30
    batch_size: int = 256
    learning_rate: float = 5e-3
    clip_norm: float = 10.0
    validation_split: float = 0.1
    log_every: int = 5


@dataclass
class TrainingArtifacts:
    model: StateBayesianNN
    guide: StateBNNGuide
    history: Dict[str, List[float]]
    action_encoder: ActionEncoder
    state_encoder: StateEncoder


def train_bnn(
    dataset: ScenarioDataset,
    *,
    action_encoder: ActionEncoder,
    state_encoder: StateEncoder,
    config: Optional[TrainingConfig] = None,
    device: Optional[torch.device] = None,
) -> TrainingArtifacts:
    config = config or TrainingConfig()
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    num_samples = len(dataset)
    val_size = max(1, int(num_samples * config.validation_split))
    train_size = num_samples - val_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size)

    input_dim = dataset.features.shape[1]
    num_classes = len(action_encoder.lookup)

    model = StateBayesianNN(input_dim, num_classes).to(device)
    guide = StateBNNGuide(model).to(device)
    optimizer = ClippedAdam({"lr": config.learning_rate, "clip_norm": config.clip_norm})
    svi = SVI(model, guide, optimizer, loss=Trace_ELBO())

    history: Dict[str, List[float]] = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, config.num_epochs + 1):
        model.train()
        epoch_loss = 0.0
        epoch_count = 0
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            loss = svi.step(batch_x, batch_y)
            epoch_loss += loss
            epoch_count += batch_x.size(0)
        epoch_loss /= epoch_count
        history["train_loss"].append(epoch_loss)

        model.eval()
        guide.eval()
        with torch.no_grad():
            val_loss = 0.0
            val_correct = 0
            val_total = 0
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                loss = svi.evaluate_loss(batch_x, batch_y)
                val_loss += loss
                log_probs = model(batch_x)
                preds = log_probs.argmax(dim=-1)
                val_correct += (preds == batch_y).sum().item()
                val_total += batch_y.size(0)
            val_loss /= max(1, val_total)
            val_acc = val_correct / max(1, val_total)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if epoch % config.log_every == 0 or epoch == 1 or epoch == config.num_epochs:
            print(
                f"Epoch {epoch:03d} | train loss {epoch_loss:.4f} | val loss {val_loss:.4f} | val acc {val_acc:.3f}"
            )

    return TrainingArtifacts(model=model, guide=guide, history=history, action_encoder=action_encoder, state_encoder=state_encoder)


In [ ]:
def mc_predict(
    artifacts: TrainingArtifacts,
    features: torch.Tensor,
    *,
    num_samples: int = 50,
    use_dropout: bool = True,
    device: Optional[torch.device] = None,
) -> Dict[str, Any]:
    device = device or next(artifacts.model.parameters()).device
    artifacts.model.to(device)
    artifacts.guide.to(device)

    if use_dropout:
        artifacts.model.backbone.train()
    else:
        artifacts.model.backbone.eval()

    predictive = pyro.infer.Predictive(
        artifacts.model,
        guide=artifacts.guide,
        num_samples=num_samples,
        return_sites=("logits",),
    )
    samples = predictive(features.to(device))
    logits = samples["logits"]  # [num_samples, batch, num_classes]
    probs = logits.softmax(dim=-1)
    mean_probs = probs.mean(dim=0)
    predictive_entropy = -(mean_probs * mean_probs.clamp(min=1e-8).log()).sum(dim=-1)
    expected_entropy = -(
        probs * probs.clamp(min=1e-8).log()
    ).sum(dim=-1).mean(dim=0)
    mutual_information = predictive_entropy - expected_entropy
    predictions = mean_probs.argmax(dim=-1)

    return {
        "mean_probs": mean_probs.detach().cpu(),
        "predictive_entropy": predictive_entropy.detach().cpu(),
        "mutual_information": mutual_information.detach().cpu(),
        "predictions": predictions.detach().cpu(),
    }


In [ ]:
def build_synthetic_dataset(
    *,
    mode: GameMode,
    num_scenarios: int,
    scenario_mix: Optional[Dict[ScenarioType, float]] = None,
    rng_seed: int = 1024,
    include_random_bot: bool = True,
) -> Tuple[ScenarioDataset, List[LabeledScenario], StateEncoder, ActionEncoder]:
    scenario_mix = scenario_mix or {
        ScenarioType.FINISHER: 0.20,
        ScenarioType.DEFENDER: 0.20,
        ScenarioType.SETUP: 0.20,
        ScenarioType.COLOR_TRAP: 0.20,
        ScenarioType.WILD_DILEMMA: 0.20,
    }
    rng = random.Random(rng_seed)
    forge = ScenarioForge(rng=rng)
    params = ScenarioParameters(scenario_type=ScenarioType.FINISHER, mode=mode)

    scenarios: List[ScenarioExample] = []
    batch_size = max(1, num_scenarios // 10)
    while len(scenarios) < num_scenarios:
        needed = min(batch_size, num_scenarios - len(scenarios))
        scenarios.extend(
            forge.generate_batch(
                scenario_mix,
                mode=mode,
                batch_size=needed,
                base_params=params,
            )
        )

    rng_seed += 1
    bots: List[BotPolicy] = [
        OracleBot(rollout_count=8, rollout_depth=4, rng=random.Random(rng_seed + 1)),
        AggressorBot(rng=random.Random(rng_seed + 2)),
        SupporterBot(rng=random.Random(rng_seed + 3)),
        ConservativeBot(rng=random.Random(rng_seed + 4)),
    ]
    if include_random_bot:
        bots.append(RandomBot(rng=random.Random(rng_seed + 5)))

    labeler = ScenarioLabeler(rng=random.Random(rng_seed + 6))
    labeled = labeler.label(scenarios, bots)

    state_encoder = StateEncoder()
    action_encoder = ActionEncoder()

    feature_rows: List[np.ndarray] = []
    label_rows: List[int] = []
    metadata: List[Dict[str, Any]] = []

    for labeled_example in labeled:
        feature_rows.append(state_encoder.encode(labeled_example))
        label_rows.append(action_encoder.encode(labeled_example))
        metadata.append(
            {
                "mode": labeled_example.scenario.state.mode.value,
                "scenario_type": labeled_example.scenario.scenario_type.value,
                "bot": labeled_example.bot_name,
                "action_token": action_encoder.decode(label_rows[-1]),
                "action_success": labeled_example.action_success,
            }
        )

    features = np.stack(feature_rows)
    labels = np.array(label_rows, dtype=np.int64)
    dataset = ScenarioDataset(features, labels, metadata)
    return dataset, labeled, state_encoder, action_encoder


In [ ]:
def export_artifacts(
    artifacts: TrainingArtifacts,
    *,
    output_dir: Path,
    model_tag: str,
    extra_metadata: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)
    param_store_path = output_dir / f"{model_tag}_param_store.pt"
    meta_path = output_dir / f"{model_tag}_meta.json"

    pyro.get_param_store().save(str(param_store_path))

    meta = {
        "model_tag": model_tag,
        "feature_size": artifacts.state_encoder.feature_size,
        "num_actions": len(artifacts.action_encoder.lookup),
        "action_tokens": artifacts.action_encoder.reverse,
        "history": artifacts.history,
    }
    if extra_metadata:
        meta.update(extra_metadata)

    with meta_path.open("w", encoding="utf-8") as fp:
        json.dump(meta, fp, indent=2, default=str)

    return {"param_store": str(param_store_path), "metadata": str(meta_path)}


In [ ]:
class BNNBot(BotPolicy):
    def __init__(
        self,
        artifacts: TrainingArtifacts,
        *,
        persona_hint: str = "OracleBot",
        rng: Optional[random.Random] = None,
        uncertainty_threshold: float = 0.25,
        mc_samples: int = 40,
    ) -> None:
        super().__init__("BNNBot", rng=rng)
        self.artifacts = artifacts
        self.persona_hint = persona_hint if persona_hint in BOT_ORDER else BOT_ORDER[0]
        self.uncertainty_threshold = uncertainty_threshold
        self.mc_samples = mc_samples

    def decide(self, engine: UnoEngine, state: GameState, player_id: str) -> BotAction:
        scenario_type = self._infer_scenario_type(state, player_id)
        metadata = self._infer_metadata(state, player_id)
        features = self.artifacts.state_encoder.encode_components(
            state=state,
            target_player=player_id,
            bot_name=self.persona_hint,
            scenario_type=scenario_type,
            metadata=metadata,
        )
        features_tensor = torch.from_numpy(features).unsqueeze(0)
        result = mc_predict(
            self.artifacts,
            features_tensor,
            num_samples=self.mc_samples,
            use_dropout=True,
        )
        mean_probs = result["mean_probs"][0]
        entropy = result["predictive_entropy"][0].item()
        mutual_info = result["mutual_information"][0].item()

        token_idx = int(result["predictions"][0].item())
        token = self.artifacts.action_encoder.decode(token_idx)
        action = self.artifacts.action_encoder.token_to_action(token, state, player_id)

        if action.action_type == ActionType.DRAW:
            # fallback to valid move with highest probability mass
            valid_actions = self.enumerate_actions(engine, state, player_id)
            ranked = []
            for candidate in valid_actions:
                candidate_token = self.artifacts.action_encoder._to_token(candidate)
                candidate_idx = self.artifacts.action_encoder.lookup.get(candidate_token)
                if candidate_idx is None:
                    continue
                ranked.append((mean_probs[candidate_idx].item(), candidate))
            if ranked:
                ranked.sort(key=lambda pair: pair[0], reverse=True)
                action = ranked[0][1]

        action.info.update(
            {
                "bnn_entropy": entropy,
                "bnn_mutual_information": mutual_info,
                "persona_hint": self.persona_hint,
                "scenario_type": scenario_type.value,
            }
        )
        return action

    def _infer_scenario_type(self, state: GameState, player_id: str) -> ScenarioType:
        hand = next(player.hand for player in state.players if player.player_id == player_id)
        if state.pending_action and state.pending_action.type == PendingActionType.DRAW_STACK:
            return ScenarioType.DEFENDER
        if len(hand) <= 2:
            return ScenarioType.FINISHER
        if any(card.type in {CardType.WILD, CardType.WILD_DRAW_FOUR} for card in hand):
            return ScenarioType.WILD_DILEMMA
        top_color = state.current_color or state.discard_pile[-1].color
        opponent_colors = self._opponent_color_counts(state, player_id)
        if top_color in opponent_colors and opponent_colors[top_color] <= 1:
            return ScenarioType.COLOR_TRAP
        return ScenarioType.SETUP

    def _infer_metadata(self, state: GameState, player_id: str) -> Dict[str, Any]:
        hand = next(player.hand for player in state.players if player.player_id == player_id)
        teammate = state.teammate_id(player_id)
        teammate_hand = next(player.hand for player in state.players if player.player_id == teammate)
        opponents = [p for p in state.players if state.team_map[p.player_id] != state.team_map[player_id]]
        opponent_colors = self._aggregate_color_counts([op.hand for op in opponents])
        weakest_color = min(opponent_colors.items(), key=lambda kv: kv[1])[0] if opponent_colors else None
        metadata = {
            "distance_to_victory": max(0, len(hand) - 1),
            "draw_stack_penalty": state.pending_action.draw_penalty if state.pending_action else 0,
            "available_wild": any(card.type in {CardType.WILD, CardType.WILD_DRAW_FOUR} for card in hand),
            "high_stakes": state.draw_stack_total >= 4,
            "teammate_preferred_color": self._dominant_color(teammate_hand),
            "weak_opponent_color": weakest_color,
        }
        return metadata

    def _dominant_color(self, hand: Sequence[Card]) -> Optional[Color]:
        counts = self._aggregate_color_counts([hand])
        if not counts:
            return None
        return max(counts.items(), key=lambda kv: kv[1])[0]

    def _aggregate_color_counts(self, hands: Sequence[Sequence[Card]]) -> Dict[Optional[Color], int]:
        counts: Dict[Optional[Color], int] = {color: 0 for color in STANDARD_COLORS}
        for hand in hands:
            for card in hand:
                if card.color in counts:
                    counts[card.color] += 1
        return counts

    def _opponent_color_counts(self, state: GameState, player_id: str) -> Dict[Color, int]:
        opponents = [p for p in state.players if state.team_map[p.player_id] != state.team_map[player_id]]
        counts = {color: 0 for color in STANDARD_COLORS}
        for player in opponents:
            for card in player.hand:
                if card.color in counts:
                    counts[card.color] += 1
        return counts


In [ ]:
def evaluate_state_with_bnn(
    artifacts: TrainingArtifacts,
    *,
    state: GameState,
    target_player: str,
    persona_name: str,
    scenario_type: ScenarioType,
    metadata: Dict[str, Any],
    num_samples: int = 40,
) -> Dict[str, Any]:
    features = artifacts.state_encoder.encode_components(
        state=state,
        target_player=target_player,
        bot_name=persona_name,
        scenario_type=scenario_type,
        metadata=metadata,
    )
    features_tensor = torch.from_numpy(features).unsqueeze(0)
    mc = mc_predict(
        artifacts,
        features_tensor,
        num_samples=num_samples,
        use_dropout=True,
    )
    return {
        "features": features,
        "mc": mc,
    }


In [ ]:
@dataclass
class ActiveLog:
    state: GameState
    target_player: str
    persona_name: str
    scenario_type: ScenarioType
    metadata: Dict[str, Any]
    action: BotAction
    resulting_state: GameState
    action_token: str
    bnn_entropy: float
    bnn_mutual_information: float


class ActiveCurriculum:
    def __init__(
        self,
        artifacts: TrainingArtifacts,
        *,
        uncertainty_threshold: float = 0.20,
        rng: Optional[random.Random] = None,
    ) -> None:
        self.artifacts = artifacts
        self.uncertainty_threshold = uncertainty_threshold
        self.engine = UnoEngine()
        self.rng = rng or random.Random()
        self.bnn_helper = BNNBot(artifacts, rng=self.rng)

    def simulate_game(
        self,
        *,
        mode: GameMode,
        bot_assignments: Dict[str, BotPolicy],
        max_turns: int = 200,
    ) -> List[ActiveLog]:
        players = [Player(player_id=pid) for pid in PLAYER_SEQUENCE]
        state = self.engine.init_game(players, mode=mode)
        logs: List[ActiveLog] = []
        turns = 0
        while not state.round_over and turns < max_turns:
            current_player = state.current_player()
            bot = bot_assignments[current_player.player_id]
            scenario_type = self.bnn_helper._infer_scenario_type(state, current_player.player_id)
            metadata = self.bnn_helper._infer_metadata(state, current_player.player_id)
            evaluation = evaluate_state_with_bnn(
                self.artifacts,
                state=state,
                target_player=current_player.player_id,
                persona_name=bot.name,
                scenario_type=scenario_type,
                metadata=metadata,
            )
            mc = evaluation["mc"]
            entropy = float(mc["predictive_entropy"][0].item())
            mutual_info = float(mc["mutual_information"][0].item())
            predicted_idx = int(mc["predictions"][0].item())
            predicted_token = self.artifacts.action_encoder.decode(predicted_idx)

            action = bot.decide(self.engine, state, current_player.player_id)
            next_state, success = self._apply_action(state, current_player.player_id, action)

            if success and (mutual_info >= self.uncertainty_threshold or entropy >= self.uncertainty_threshold):
                logs.append(
                    ActiveLog(
                        state=state,
                        target_player=current_player.player_id,
                        persona_name=bot.name,
                        scenario_type=scenario_type,
                        metadata=metadata,
                        action=action,
                        resulting_state=next_state,
                        action_token=self.artifacts.action_encoder._to_token(action),
                        bnn_entropy=entropy,
                        bnn_mutual_information=mutual_info,
                    )
                )

            state = next_state
            turns += 1
        return logs

    def _apply_action(self, state: GameState, player_id: str, action: BotAction) -> Tuple[GameState, bool]:
        try:
            if action.action_type == ActionType.PLAY and action.card is not None:
                return self.engine.play_card(state, player_id, action.card, action.chosen_color), True
            if action.action_type == ActionType.DRAW:
                return self.engine.draw_card(state, player_id), True
            if action.action_type == ActionType.PASS:
                return self.engine.pass_turn(state, player_id), True
        except InvalidMoveError:
            return state, False
        return state, True


In [ ]:
def build_dataset_from_labeled(
    labeled: Sequence[LabeledScenario],
    *,
    state_encoder: StateEncoder,
    action_encoder: ActionEncoder,
) -> ScenarioDataset:
    feature_rows: List[np.ndarray] = []
    label_rows: List[int] = []
    metadata: List[Dict[str, Any]] = []

    for labeled_example in labeled:
        feature_rows.append(state_encoder.encode(labeled_example))
        label_rows.append(action_encoder.encode(labeled_example))
        metadata.append(
            {
                "mode": labeled_example.scenario.state.mode.value,
                "scenario_type": labeled_example.scenario.scenario_type.value,
                "bot": labeled_example.bot_name,
                "action_token": action_encoder.decode(label_rows[-1]),
                "action_success": labeled_example.action_success,
            }
        )

    features = np.stack(feature_rows)
    labels = np.array(label_rows, dtype=np.int64)
    return ScenarioDataset(features, labels, metadata)


## Running the Curriculum

Execute the following cells to generate the synthetic dataset, train the initial BNN (v0.1), run the active learning loop, and export checkpoints for both Classic 2v2 and Go Wild 2v2. Expect the full loop to take several minutes on CPU; using a GPU runtime is recommended.

In [ ]:
def run_curriculum_loop(
    *,
    mode: GameMode,
    initial_scenarios: int = 5000,
    iterations: int = 2,
    games_per_iteration: int = 10,
    uncertainty_threshold: float = 0.25,
    epsilon: float = 0.005,
    rng_seed: int = 7,
) -> Tuple[TrainingArtifacts, List[LabeledScenario]]:
    rng = random.Random(rng_seed)
    dataset, labeled, state_encoder, action_encoder = build_synthetic_dataset(
        mode=mode,
        num_scenarios=initial_scenarios,
        rng_seed=rng_seed,
    )

    pyro.clear_param_store()
    artifacts = train_bnn(
        dataset,
        action_encoder=action_encoder,
        state_encoder=state_encoder,
        config=TrainingConfig(num_epochs=18, batch_size=96, learning_rate=2.5e-3),
    )

    labeler = ScenarioLabeler(rng=rng)

    for iteration in range(1, iterations + 1):
        curriculum = ActiveCurriculum(
            artifacts,
            uncertainty_threshold=uncertainty_threshold,
            rng=rng,
        )
        logs: List[ActiveLog] = []
        for game_index in range(games_per_iteration):
            bot_assignments = {
                "P0": AggressorBot(rng=rng),
                "P1": SupporterBot(rng=rng),
                "P2": ConservativeBot(rng=rng),
                "P3": RandomBot(rng=rng),
            }
            logs.extend(
                curriculum.simulate_game(
                    mode=mode,
                    bot_assignments=bot_assignments,
                )
            )
        if not logs:
            print(f"Iteration {iteration}: no high-uncertainty scenarios captured.")
            continue

        new_labeled = labeler.logs_to_labeled(logs)
        labeled.extend(new_labeled)
        dataset = build_dataset_from_labeled(
            labeled,
            state_encoder=state_encoder,
            action_encoder=action_encoder,
        )

        pyro.clear_param_store()
        new_artifacts = train_bnn(
            dataset,
            action_encoder=action_encoder,
            state_encoder=state_encoder,
            config=TrainingConfig(num_epochs=12, batch_size=96, learning_rate=1.8e-3),
        )

        previous_val_loss = artifacts.history["val_loss"][-1]
        new_val_loss = new_artifacts.history["val_loss"][-1]
        improvement = previous_val_loss - new_val_loss
        print(
            f"Iteration {iteration}: captured {len(logs)} scenarios | val improvement {improvement:.4f}"
        )
        artifacts = new_artifacts
        if improvement < epsilon:
            print("Stopping early due to marginal improvement threshold.")
            break

    artifacts.action_encoder = action_encoder
    artifacts.state_encoder = state_encoder
    return artifacts, labeled


In [ ]:
OUTPUT_DIR = Path("models")

classic_artifacts, classic_labeled = run_curriculum_loop(
    mode=GameMode.CLASSIC_2V2,
    initial_scenarios=5000,
    iterations=2,
    games_per_iteration=10,
    uncertainty_threshold=0.23,
)
classic_paths = export_artifacts(
    classic_artifacts,
    output_dir=OUTPUT_DIR,
    model_tag="classic_2v2_v0_2",
    extra_metadata={
        "mode": GameMode.CLASSIC_2V2.value,
        "total_examples": len(classic_labeled),
    },
)
print("Classic model saved:", classic_paths)

pyro.clear_param_store()

go_wild_artifacts, go_wild_labeled = run_curriculum_loop(
    mode=GameMode.GO_WILD_2V2,
    initial_scenarios=6000,
    iterations=3,
    games_per_iteration=12,
    uncertainty_threshold=0.27,
)
go_wild_paths = export_artifacts(
    go_wild_artifacts,
    output_dir=OUTPUT_DIR,
    model_tag="go_wild_2v2_v0_2",
    extra_metadata={
        "mode": GameMode.GO_WILD_2V2.value,
        "total_examples": len(go_wild_labeled),
    },
)
print("Go Wild model saved:", go_wild_paths)

### Inference Helpers

Once checkpoints are saved, reload the Pyro parameter store and reconstruct the network using `StateBayesianNN`/`StateBNNGuide`. Use `evaluate_state_with_bnn` to obtain posterior predictive statistics for arbitrary UNO states. During gameplay, plug the exported encoders and action registry into agent code to translate model outputs back into UNO actions.